# Cost Model — smoke test

Confirms the container is wired up correctly and demonstrates why this project
selects models by **money**, not accuracy.


In [1]:
import numpy as np
from fraud_cost import (
    total_cost, cost_curve, optimal_threshold,
    policy_e_predict, best_amount_baseline,
)
print("fraud_cost imported OK")

fraud_cost imported OK


## 1. The objective

`TotalCost = c_review x (TP + FP) + sum(Amount over FN)`

Review cost per alert raised, plus money lost to fraud that slipped through.

In [2]:
y_true  = np.array([1, 1, 0, 0])
y_pred  = np.array([1, 0, 1, 0])
amounts = np.array([100.0, 500.0, 50.0, 20.0])

cost = total_cost(y_true, y_pred, amounts, c_review=3.0)
print(f"TP+FP = 2 alerts x EUR3 = EUR6, plus one missed EUR500 fraud")
print(f"total cost = EUR{cost:,.2f}")

TP+FP = 2 alerts x EUR3 = EUR6, plus one missed EUR500 fraud
total cost = EUR506.00


## 2. Why a global threshold is the wrong shape

Minimising that objective gives a **per-transaction** rule:

`alert iff p_i x Amount_i > c_review`

Watch what it does to two rows that a global threshold gets backwards.

In [3]:
probabilities = np.array([0.01, 0.50, 0.001, 0.90])
amounts_e     = np.array([1000.0, 4.0, 10000.0, 1.0])

alerts = policy_e_predict(probabilities, amounts_e, c_review=3.0)

print(f"{'p':>7} {'Amount':>10} {'p*Amount':>10} {'alert':>7}")
for p, a, fl in zip(probabilities, amounts_e, alerts):
    print(f"{p:>7.3f} {a:>10,.0f} {p*a:>10.2f} {fl:>7}")
print()
print("Row 0: p=0.01 on EUR1,000 -> ALERTS  (low probability, real money)")
print("Row 3: p=0.90 on EUR1     -> ignored (high probability, trivial money)")
print("A global threshold does exactly the opposite on these two rows.")

      p     Amount   p*Amount   alert
  0.010      1,000      10.00       1
  0.500          4       2.00       0
  0.001     10,000      10.00       1
  0.900          1       0.90       0

Row 0: p=0.01 on EUR1,000 -> ALERTS  (low probability, real money)
Row 3: p=0.90 on EUR1     -> ignored (high probability, trivial money)
A global threshold does exactly the opposite on these two rows.


## 3. Tie safety

`RandomForest(n_estimators=100)` emits only ~101 distinct probabilities, so many
rows share a boundary score. A rank-based "top k" sweep reports optima inside a
tied block that **no threshold can reach** — silently.

All three scores below tie, so only two alert sets are reachable.

In [4]:
scores   = np.array([0.5, 0.5, 0.5])
y_tied   = np.array([1, 0, 0])
amt_tied = np.array([1000.0, 10.0, 10.0])

thresholds, costs = cost_curve(y_tied, scores, amt_tied, c_review=3.0)
for t, c in zip(thresholds, costs):
    print(f"threshold {t:>6} -> cost EUR{c:>8,.2f}")
print()
print(f"achievable minimum = EUR{costs.min():,.2f}")
print("a naive rank sweep would report EUR3.00 - a threshold that cannot exist")

threshold    inf -> cost EUR1,000.00
threshold    0.5 -> cost EUR    9.00

achievable minimum = EUR9.00
a naive rank sweep would report EUR3.00 - a threshold that cannot exist


## 4. Optimal threshold and the no-ML baseline

The alert rate matters as much as the threshold: a probability cut does not
survive a refit, but *"flag the top X%"* does.

In [5]:
scores2  = np.array([0.9, 0.8, 0.7, 0.6])
y2       = np.array([1, 0, 1, 0])
amounts2 = np.array([100.0, 10.0, 200.0, 10.0])

t, c, rate = optimal_threshold(y2, scores2, amounts2, c_review=3.0)
print(f"optimal threshold : {t}")
print(f"cost              : EUR{c:,.2f}")
print(f"alert rate        : {rate:.0%}")

x, base = best_amount_baseline(y2, amounts2, c_review=3.0)
print(f"\nno-ML baseline    : review every transaction >= EUR{x:,.0f}, cost EUR{base:,.2f}")
print("If no model beats this, that IS the finding.")

optimal threshold : 0.7
cost              : EUR9.00
alert rate        : 75%

no-ML baseline    : review every transaction >= EUR100, cost EUR6.00
If no model beats this, that IS the finding.


## 5. Full test suite

```bash
./run.sh test
```
